# **Rilevamento di Siti Web di Phishing tramite Machine Learning**

Questo progetto applica una pipeline di Machine Learning al dataset **Phishing Websites** con l'obiettivo di classificare i siti web come **legittimi** o **phishing** sulla base di 30 feature estratte dalla struttura degli URL e dalle caratteristiche dei siti.

Il dataset è distribuito in formato **ARFF** (Attribute-Relation File Format) e tutte le feature sono codificate con valori discreti:
- **-1**: indica una caratteristica associata ai siti di phishing
- **0**: indica un caso intermedio/sospetto
- **1**: indica una caratteristica associata ai siti legittimi

La variabile target `Result` assume valore **1** (legittimo) o **-1** (phishing).

## **Struttura del Progetto**
1. Caricamento e ispezione del dataset
2. Analisi Esplorativa dei Dati (EDA)
3. **Data Cleaning** (fase principale)
4. Feature Engineering
5. Bilanciamento del dataset
6. Train-Test Split
7. Addestramento e Valutazione dei Modelli
8. Conclusioni

## **1. Import delle Librerie**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from collections import Counter

import warnings
warnings.filterwarnings('ignore')

# Stile grafici
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

## **2. Caricamento del Dataset**

Il dataset è in formato ARFF. Dopo il caricamento tramite `scipy.io.arff`, i valori sono decodificati da byte a stringa e poi convertiti in interi, poiché tutte le feature sono categoriche ordinate numericamente.

In [ ]:
# Caricamento del dataset ARFF
data, meta = arff.loadarff('dataset/Training Dataset.arff')
df_raw = pd.DataFrame(data)

# Decodifica bytes -> str -> int
for col in df_raw.columns:
    df_raw[col] = df_raw[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
df_raw = df_raw.astype(int)

print(f'Dimensioni dataset grezzo: {df_raw.shape}')
print(f'Feature: {df_raw.shape[1] - 1}  |  Righe: {df_raw.shape[0]}')
df_raw.head()

## **3. Analisi Esplorativa dei Dati (EDA)**

Prima di procedere con il cleaning, esploriamo il dataset per capire:
- La distribuzione della variabile target
- La presenza di valori mancanti
- La presenza di righe duplicate
- La distribuzione delle singole feature

In [ ]:
# Panoramica generale
print('=== INFORMAZIONI GENERALI ===')
df_raw.info()
print()
print('=== STATISTICHE DESCRITTIVE ===')
df_raw.describe()

In [ ]:
# Distribuzione del target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

target_counts = df_raw['Result'].value_counts()
labels = ['Legittimo (1)', 'Phishing (-1)']
colors = ['#4CAF50', '#F44336']

axes[0].bar(labels, [target_counts[1], target_counts[-1]], color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Distribuzione della variabile target: Result', fontsize=13)
axes[0].set_ylabel('Numero di campioni')
for i, v in enumerate([target_counts[1], target_counts[-1]]):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].pie(
    [target_counts[1], target_counts[-1]],
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
axes[1].set_title('Proporzione Legittimo / Phishing (grezzo)', fontsize=13)

plt.tight_layout()
plt.show()

print(f"Legittimo  (1) : {target_counts[1]} campioni ({target_counts[1]/len(df_raw)*100:.1f}%)")
print(f"Phishing  (-1) : {target_counts[-1]} campioni ({target_counts[-1]/len(df_raw)*100:.1f}%)")

In [ ]:
# Mappa dei valori mancanti
missing = df_raw.isnull().sum()
print('=== VALORI MANCANTI PER COLONNA ===')
print(missing[missing > 0] if missing.sum() > 0 else 'Nessun valore mancante rilevato nel dataset grezzo.')

plt.figure(figsize=(14, 4))
sns.heatmap(df_raw.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Mappa dei Valori Mancanti', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Analisi dei duplicati
n_dup = df_raw.duplicated().sum()
print(f'Righe duplicate nel dataset grezzo: {n_dup} ({n_dup/len(df_raw)*100:.1f}% del totale)')

In [ ]:
# Distribuzione delle feature (valori unici per colonna)
print('=== DOMINIO DEI VALORI PER FEATURE ===')
feature_domains = {}
for col in df_raw.columns:
    feature_domains[col] = sorted(df_raw[col].unique().tolist())
    print(f'  {col:35s}: {feature_domains[col]}')

In [ ]:
# Heatmap della correlazione (dataset grezzo)
plt.figure(figsize=(16, 12))
corr_matrix = df_raw.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, cbar_kws={'shrink': 0.8},
    annot_kws={'size': 7}
)
plt.title('Matrice di Correlazione – Dataset Grezzo', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Distribuzione delle feature rispetto al target
features = [c for c in df_raw.columns if c != 'Result']
n_cols = 5
n_rows = -(-len(features) // n_cols)  # ceil division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))
axes = axes.flatten()

for i, feat in enumerate(features):
    ct = df_raw.groupby([feat, 'Result']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=axes[i], color=['#F44336', '#4CAF50'], edgecolor='black', linewidth=0.5, width=0.7)
    axes[i].set_title(feat, fontsize=9, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count', fontsize=8)
    axes[i].tick_params(axis='x', rotation=0, labelsize=8)
    axes[i].legend(title='Result', labels=['Phishing (-1)', 'Legittimo (1)'], fontsize=7)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Distribuzione delle Feature rispetto al Target (Result)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## **4. Data Cleaning**

Questa è la fase principale del progetto. Il processo di pulizia dei dati si articola nei seguenti passi:

| Step | Tecnica | Motivazione |
|------|---------|-------------|
| 4.1 | Validazione del dominio dei valori | Verifica che ogni feature contenga solo valori ammessi secondo lo schema ARFF |
| 4.2 | Rimozione dei duplicati | 5206 righe duplicate (~47%) distorcono le distribuzioni e possono causare data leakage |
| 4.3 | Verifica e gestione dei valori mancanti | I NaN non sono evidenti ma potrebbero nascondersi come valori anomali |
| 4.4 | Rilevamento e gestione degli outlier | Con valori discreti, gli outlier si manifestano come violazioni del dominio |
| 4.5 | Verifica della consistenza inter-feature | Controllo di inconsistenze logiche tra feature correlate |
| 4.6 | Analisi della varianza (feature costanti o quasi) | Feature con varianza ~0 non apportano informazione predittiva |
| 4.7 | Analisi della correlazione | Rimozione di feature altamente correlate per ridurre la ridondanza |
| 4.8 | Report finale del cleaning | Riepilogo di tutte le operazioni effettuate |

In [ ]:
# Lavoriamo su una copia del dataset grezzo
df = df_raw.copy()
cleaning_log = []  # Log delle operazioni di cleaning

print(f'Dataset grezzo: {df.shape[0]} righe x {df.shape[1]} colonne')

### **4.1 – Validazione del Dominio dei Valori**

Ogni feature ha un dominio definito nello schema ARFF. Verifichiamo che non ci siano valori fuori dal dominio atteso.

In [ ]:
# Domini attesi per ciascuna feature (da schema ARFF)
expected_domains = {
    'having_IP_Address': {-1, 1},
    'URL_Length': {-1, 0, 1},
    'Shortining_Service': {-1, 1},
    'having_At_Symbol': {-1, 1},
    'double_slash_redirecting': {-1, 1},
    'Prefix_Suffix': {-1, 1},
    'having_Sub_Domain': {-1, 0, 1},
    'SSLfinal_State': {-1, 0, 1},
    'Domain_registeration_length': {-1, 1},
    'Favicon': {-1, 1},
    'port': {-1, 1},
    'HTTPS_token': {-1, 1},
    'Request_URL': {-1, 1},
    'URL_of_Anchor': {-1, 0, 1},
    'Links_in_tags': {-1, 0, 1},
    'SFH': {-1, 0, 1},
    'Submitting_to_email': {-1, 1},
    'Abnormal_URL': {-1, 1},
    'Redirect': {0, 1},
    'on_mouseover': {-1, 1},
    'RightClick': {-1, 1},
    'popUpWidnow': {-1, 1},
    'Iframe': {-1, 1},
    'age_of_domain': {-1, 1},
    'DNSRecord': {-1, 1},
    'web_traffic': {-1, 0, 1},
    'Page_Rank': {-1, 1},
    'Google_Index': {-1, 1},
    'Links_pointing_to_page': {-1, 0, 1},
    'Statistical_report': {-1, 1},
    'Result': {-1, 1},
}

print('=== VALIDAZIONE DOMINIO DEI VALORI ===')
violations_found = False
violation_rows_all = set()

for col, domain in expected_domains.items():
    actual_values = set(df[col].unique())
    invalid_values = actual_values - domain
    if invalid_values:
        violations_found = True
        mask = ~df[col].isin(domain)
        n_violations = mask.sum()
        violation_rows_all.update(df[mask].index.tolist())
        print(f'  ❌ {col}: valori non validi {invalid_values} ({n_violations} righe)')
    else:
        print(f'  ✅ {col}: dominio corretto {sorted(domain)}')

if not violations_found:
    print('\n✅ Nessuna violazione del dominio rilevata. Tutti i valori sono conformi allo schema ARFF.')
else:
    print(f'\n⚠️  Totale righe con violazioni: {len(violation_rows_all)}')
    cleaning_log.append(f'Step 4.1 – Rimosse {len(violation_rows_all)} righe con violazioni del dominio')
    df = df[~df.index.isin(violation_rows_all)].reset_index(drop=True)

### **4.2 – Rimozione dei Duplicati**

Il dataset grezzo contiene **5206 righe duplicate** (~47% del totale). Questo è un problema critico:
- Distorce le distribuzioni delle classi
- Introduce **data leakage** se record identici finiscono in train e test
- Può portare a overfitting sui modelli

In [ ]:
n_before = len(df)
n_dup = df.duplicated().sum()

print(f'Righe prima della deduplicazione: {n_before}')
print(f'Righe duplicate rilevate:         {n_dup} ({n_dup/n_before*100:.1f}%)')

# Distribuzione target prima della deduplicazione
print(f'\nDistribuzione target PRIMA della deduplicazione:')
vc_before = df['Result'].value_counts()
print(f'  Legittimo  (1): {vc_before.get(1, 0)} ({vc_before.get(1, 0)/n_before*100:.1f}%)')
print(f'  Phishing  (-1): {vc_before.get(-1, 0)} ({vc_before.get(-1, 0)/n_before*100:.1f}%)')

# Rimozione duplicati
df = df.drop_duplicates().reset_index(drop=True)
n_after = len(df)

print(f'\nRighe dopo la deduplicazione:     {n_after}')
print(f'Righe rimosse:                    {n_before - n_after}')

# Distribuzione target dopo la deduplicazione
print(f'\nDistribuzione target DOPO la deduplicazione:')
vc_after = df['Result'].value_counts()
print(f'  Legittimo  (1): {vc_after.get(1, 0)} ({vc_after.get(1, 0)/n_after*100:.1f}%)')
print(f'  Phishing  (-1): {vc_after.get(-1, 0)} ({vc_after.get(-1, 0)/n_after*100:.1f}%)')

cleaning_log.append(f'Step 4.2 – Rimosse {n_before - n_after} righe duplicate ({n_before - n_after}/{n_before} = {(n_before-n_after)/n_before*100:.1f}%)')

# Visualizzazione confronto
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = ['Legittimo (1)', 'Phishing (-1)']
colors = ['#4CAF50', '#F44336']

vals_before = [vc_before.get(1, 0), vc_before.get(-1, 0)]
vals_after  = [vc_after.get(1, 0),  vc_after.get(-1, 0)]

axes[0].bar(labels, vals_before, color=colors, edgecolor='black')
axes[0].set_title(f'Prima della deduplicazione (n={n_before})', fontsize=11)
axes[0].set_ylabel('Campioni')
for i, v in enumerate(vals_before):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].bar(labels, vals_after, color=colors, edgecolor='black')
axes[1].set_title(f'Dopo la deduplicazione (n={n_after})', fontsize=11)
axes[1].set_ylabel('Campioni')
for i, v in enumerate(vals_after):
    axes[1].text(i, v + 20, str(v), ha='center', fontweight='bold')

plt.suptitle('Effetto della Rimozione dei Duplicati sulla Distribuzione del Target', fontsize=12)
plt.tight_layout()
plt.show()

### **4.3 – Verifica e Gestione dei Valori Mancanti**

Anche se i NaN classici non sono presenti nel dataset, è importante verificare se vi sono valori sentinella (come `0` in feature binarie {-1,1}) che potrebbero rappresentare dati mancanti.

In [ ]:
# Verifica NaN classici
nan_counts = df.isnull().sum()
print('=== VERIFICA VALORI MANCANTI (NaN) ===')
if nan_counts.sum() == 0:
    print('✅ Nessun valore NaN presente nel dataset.')
else:
    print(nan_counts[nan_counts > 0])

print()

# Analisi del valore '0' nelle feature binarie {-1, 1}
# In queste feature, 0 non è un valore semanticamente definito -> possibile encoding di 'missing'
binary_features = [col for col, dom in expected_domains.items() if dom == {-1, 1} and col != 'Result']
print('=== ANALISI VALORE "0" NELLE FEATURE BINARIE {-1, 1} ===')
print('(In queste feature, 0 non è definito nello schema e potrebbe rappresentare un dato mancante/incerto)\n')

zeros_in_binary = {}
for col in binary_features:
    n_zeros = (df[col] == 0).sum()
    if n_zeros > 0:
        zeros_in_binary[col] = n_zeros
        print(f'  ⚠️  {col}: {n_zeros} valori pari a 0')

if not zeros_in_binary:
    print('  ✅ Nessun valore 0 nelle feature binarie. Nessuna azione necessaria.')
else:
    print(f'\nQueste feature non sono nella lista binaria – i valori 0 sono già stati validati in 4.1')

print()

# Percentuale di valori '0' (caso sospetto) nelle feature a 3 valori {-1,0,1}
ternary_features = [col for col, dom in expected_domains.items() if dom == {-1, 0, 1} and col != 'Result']
print('=== DISTRIBUZIONE DEL VALORE "0" NELLE FEATURE TERNARIE {-1, 0, 1} ===')
for col in ternary_features:
    n_zero = (df[col] == 0).sum()
    pct = n_zero / len(df) * 100
    print(f'  {col:35s}: {n_zero:5d} valori 0 ({pct:.1f}%)')

cleaning_log.append('Step 4.3 – Nessun NaN rilevato. I valori 0 nelle feature ternarie sono semanticamente validi.')

### **4.4 – Rilevamento e Gestione degli Outlier**

Trattandosi di feature categoriche con valori discreti in {-1, 0, 1}, gli outlier non si manifestano come valori numerici estremi, ma come **violazioni del dominio** (già gestite nel passo 4.1). 

Tuttavia, effettuiamo un'analisi statistica per identificare eventuali sbilanciamenti critici all'interno di ciascuna feature.

In [ ]:
# Analisi della distribuzione di frequenza per ogni feature
print('=== DISTRIBUZIONE DI FREQUENZA DELLE FEATURE ===' )
print(f'{"Feature":<35} {"Valore":>8} {"Count":>8} {"% Totale":>10}')
print('-' * 65)

dominant_features = []  # Feature con un valore dominante > 90%

for col in df.columns:
    vc = df[col].value_counts(normalize=True).sort_index()
    for val, pct in vc.items():
        count = (df[col] == val).sum()
        flag = ' ⚠️' if pct > 0.90 else ''
        print(f'{col:<35} {str(val):>8} {count:>8} {pct*100:>9.1f}%{flag}')
    if vc.max() > 0.90:
        dominant_features.append((col, vc.idxmax(), vc.max()))
    print()

if dominant_features:
    print('\n=== FEATURE CON VALORE DOMINANTE (> 90%) ===')
    for feat, val, pct in dominant_features:
        print(f'  ⚠️  {feat}: valore {val} occorre nel {pct*100:.1f}% dei casi')
else:
    print('\n✅ Nessuna feature con valore dominante superiore al 90%.')

cleaning_log.append(f'Step 4.4 – Analisi outlier completata. Feature con valore dominante (>90%): {[f[0] for f in dominant_features] if dominant_features else "nessuna"}')

### **4.5 – Verifica della Consistenza Inter-Feature**

Alcune feature del dataset sono logicamente correlate. Controlliamo se esistono combinazioni di valori che risultano inconsistenti dal punto di vista semantico.

In [ ]:
# Regole di consistenza semantica note nel dominio phishing:
# Regola 1: Se 'having_IP_Address' == -1 (IP nell'URL, tipico di phishing),
#            allora 'Domain_registeration_length' non dovrebbe essere 1 (dominio registrato a lungo termine)
# Regola 2: Se 'SSLfinal_State' == 1 (SSL valido), 'HTTPS_token' non dovrebbe essere -1 (HTTPS assente)
# Regola 3: Se 'Google_Index' == -1 (non indicizzato), 'web_traffic' non dovrebbe essere 1 (alto traffico)

print('=== VERIFICA CONSISTENZA SEMANTICA TRA FEATURE ===\n')

inconsistencies = {}

# Regola 1
rule1_mask = (df['having_IP_Address'] == -1) & (df['Domain_registeration_length'] == 1)
n_r1 = rule1_mask.sum()
inconsistencies['R1'] = n_r1
print(f'Regola 1 – IP nell\'URL + Dominio registrato a lungo termine:')
print(f'  → {n_r1} campioni potenzialmente inconsistenti ({n_r1/len(df)*100:.2f}%)')
print(f'  Nota: può verificarsi (es. IP statico su dominio legittimo) – non rimossi automaticamente\n')

# Regola 2
rule2_mask = (df['SSLfinal_State'] == 1) & (df['HTTPS_token'] == -1)
n_r2 = rule2_mask.sum()
inconsistencies['R2'] = n_r2
print(f'Regola 2 – SSL valido + Token HTTPS assente nell\'URL:')
print(f'  → {n_r2} campioni potenzialmente inconsistenti ({n_r2/len(df)*100:.2f}%)')
print(f'  Nota: condizione ambigua – non rimossi automaticamente\n')

# Regola 3
rule3_mask = (df['Google_Index'] == -1) & (df['web_traffic'] == 1)
n_r3 = rule3_mask.sum()
inconsistencies['R3'] = n_r3
print(f'Regola 3 – Non indicizzato da Google + Alto traffico web:')
print(f'  → {n_r3} campioni potenzialmente inconsistenti ({n_r3/len(df)*100:.2f}%)')
print(f'  Nota: combinazione molto sospetta, ma potrebbe essere lag temporale – non rimossi automaticamente\n')

# Visualizzazione
fig, ax = plt.subplots(figsize=(7, 4))
rule_labels = [
    'R1: IP + Dominio\nregistrato a lungo',
    'R2: SSL valido +\nHTTPS token assente',
    'R3: Non indicizzato\n+ Alto traffico'
]
rule_counts = [n_r1, n_r2, n_r3]
bars = ax.barh(rule_labels, rule_counts, color=['#FF9800', '#FF9800', '#FF5722'], edgecolor='black')
ax.set_xlabel('Numero di campioni')
ax.set_title('Campioni con possibili inconsistenze semantiche', fontsize=12)
for bar, count in zip(bars, rule_counts):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, str(count), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

cleaning_log.append(f'Step 4.5 – Verifica consistenza: R1={n_r1}, R2={n_r2}, R3={n_r3} campioni sospetti (conservati per analisi successiva)')

### **4.6 – Analisi della Varianza (Feature Quasi-Costanti)**

Feature con varianza molto bassa (o nulla) non contribuiscono alla capacità predittiva del modello e vengono rimosse.

In [ ]:
# Calcolo della varianza per ogni feature
feature_cols = [c for c in df.columns if c != 'Result']
variances = df[feature_cols].var().sort_values()

print('=== VARIANZA DELLE FEATURE (ordine crescente) ===')
for col, var in variances.items():
    flag = ' ⚠️  QUASI-COSTANTE' if var < 0.05 else ''
    print(f'  {col:<35}: {var:.4f}{flag}')

# Soglia per feature quasi-costanti
VAR_THRESHOLD = 0.05
low_var_features = variances[variances < VAR_THRESHOLD].index.tolist()

print(f'\n=== FEATURE CON VARIANZA < {VAR_THRESHOLD} ===')
if low_var_features:
    for feat in low_var_features:
        print(f'  ❌ Rimossa: {feat} (varianza = {variances[feat]:.4f})')
    df = df.drop(columns=low_var_features)
    cleaning_log.append(f'Step 4.6 – Rimosse {len(low_var_features)} feature quasi-costanti: {low_var_features}')
else:
    print(f'  ✅ Nessuna feature con varianza inferiore alla soglia {VAR_THRESHOLD}.')
    cleaning_log.append(f'Step 4.6 – Nessuna feature quasi-costante rilevata (soglia={VAR_THRESHOLD}).')

# Visualizzazione varianze
plt.figure(figsize=(14, 5))
colors_var = ['#F44336' if v < VAR_THRESHOLD else '#2196F3' for v in variances.values]
bars = plt.bar(variances.index, variances.values, color=colors_var, edgecolor='black', linewidth=0.5)
plt.axhline(y=VAR_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Soglia = {VAR_THRESHOLD}')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.ylabel('Varianza')
plt.title('Varianza delle Feature (rosso = sotto soglia)', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

### **4.7 – Analisi della Correlazione e Rimozione Feature Ridondanti**

Feature fortemente correlate tra loro sono ridondanti: forniscono informazioni simili al modello aumentando la complessità senza migliorare le prestazioni. Rimuoviamo una delle due feature in ogni coppia con correlazione |r| > 0.85.

In [ ]:
# Calcolo matrice di correlazione sulle sole feature (post cleaning)
feature_cols_clean = [c for c in df.columns if c != 'Result']
corr_matrix = df[feature_cols_clean].corr().abs()

# Individuazione coppie con correlazione alta
CORR_THRESHOLD = 0.85
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(col, row, upper_tri.loc[row, col])
                   for col in upper_tri.columns
                   for row in upper_tri.index
                   if pd.notna(upper_tri.loc[row, col]) and upper_tri.loc[row, col] > CORR_THRESHOLD]

print(f'=== COPPIE DI FEATURE CON CORRELAZIONE |r| > {CORR_THRESHOLD} ===')
to_drop_corr = set()

if high_corr_pairs:
    for f1, f2, corr_val in sorted(high_corr_pairs, key=lambda x: -x[2]):
        # Manteniamo la feature più correlata con il target
        corr_f1_target = abs(df[f1].corr(df['Result']))
        corr_f2_target = abs(df[f2].corr(df['Result']))
        drop_col = f2 if corr_f1_target >= corr_f2_target else f1
        keep_col = f1 if drop_col == f2 else f2
        to_drop_corr.add(drop_col)
        print(f'  |r|={corr_val:.3f}  {f1} ↔ {f2}')
        print(f'    → Rimuovo: {drop_col} (corr. con target={min(corr_f1_target, corr_f2_target):.3f})')
        print(f'    → Mantengo: {keep_col} (corr. con target={max(corr_f1_target, corr_f2_target):.3f})\n')
    
    if to_drop_corr:
        df = df.drop(columns=list(to_drop_corr))
        cleaning_log.append(f'Step 4.7 – Rimosse {len(to_drop_corr)} feature ad alta correlazione (soglia={CORR_THRESHOLD}): {list(to_drop_corr)}')
else:
    print(f'  ✅ Nessuna coppia di feature con correlazione superiore alla soglia {CORR_THRESHOLD}.')
    cleaning_log.append(f'Step 4.7 – Nessuna feature ridondante rilevata (soglia correlazione={CORR_THRESHOLD}).')

print(f'\nFeature rimanenti dopo il cleaning: {len([c for c in df.columns if c != "Result"])}')

# Correlazione delle feature rimaste con il target
corr_with_target = df.corr()['Result'].drop('Result').sort_values(key=abs, ascending=False)
plt.figure(figsize=(12, 5))
colors_target = ['#4CAF50' if v > 0 else '#F44336' for v in corr_with_target.values]
plt.bar(corr_with_target.index, corr_with_target.values, color=colors_target, edgecolor='black', linewidth=0.5)
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.ylabel('Correlazione con Result')
plt.title('Correlazione delle Feature con il Target (Result)', fontsize=12)
plt.tight_layout()
plt.show()

### **4.8 – Report Finale del Data Cleaning**

In [ ]:
print('=' * 65)
print('           REPORT FINALE – DATA CLEANING')
print('=' * 65)
print(f'\nDataset grezzo:  {df_raw.shape[0]:6d} righe  x  {df_raw.shape[1]:2d} feature')
print(f'Dataset pulito:  {df.shape[0]:6d} righe  x  {df.shape[1]:2d} feature (target incluso)')
print(f'\nRighe rimosse:   {df_raw.shape[0] - df.shape[0]:6d}  ({(df_raw.shape[0]-df.shape[0])/df_raw.shape[0]*100:.1f}%)')
print(f'Feature rimosse: {df_raw.shape[1] - df.shape[1]:6d}')
print()
print('Operazioni eseguite:')
for i, log in enumerate(cleaning_log, 1):
    print(f'  {i}. {log}')
print()
print('Distribuzione finale del target:')
vc_final = df['Result'].value_counts()
print(f'  Legittimo  (1): {vc_final.get(1, 0):5d} ({vc_final.get(1, 0)/len(df)*100:.1f}%)')
print(f'  Phishing  (-1): {vc_final.get(-1, 0):5d} ({vc_final.get(-1, 0)/len(df)*100:.1f}%)')
print()
print('Feature nel dataset pulito:')
print([c for c in df.columns if c != 'Result'])
print('=' * 65)

df_clean = df.copy()
print('\nPrime righe del dataset pulito:')
df_clean.head()

## **5. Feature Engineering**

Dopo il cleaning, separiamo feature e target e applichiamo la standardizzazione per i modelli che ne hanno bisogno (KNN, Logistic Regression).

In [ ]:
# Separazione feature e target
target = 'Result'
X = df_clean.drop(columns=[target])
y = df_clean[target]

print(f'Feature (X): {X.shape[1]} colonne')
print(f'Target  (y): distribuzione → {dict(y.value_counts())}')

## **6. Train-Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:  {X_train.shape[0]} campioni')
print(f'Test set:      {X_test.shape[0]} campioni')
print(f'\nDistribuzione y_train: {dict(y_train.value_counts())}')
print(f'Distribuzione y_test:  {dict(y_test.value_counts())}')

## **7. Bilanciamento del Dataset di Training con SMOTE**

Anche dopo il cleaning il dataset presenta uno sbilanciamento moderato. Applichiamo SMOTE **solo sul training set** per evitare data leakage.

In [ ]:
print(f'Distribuzione classi PRIMA di SMOTE: {Counter(y_train)}')

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Distribuzione classi DOPO SMOTE:     {Counter(y_train_res)}')

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

print(f'\nScaling applicato: StandardScaler (fit su training, transform su test)')

## **8. Addestramento e Valutazione dei Modelli**

Addestriamo tre modelli di classificazione:
- **Decision Tree** (non richiede scaling)
- **Logistic Regression**
- **K-Nearest Neighbors**

Valutiamo ciascun modello con: Accuracy, Precision, Recall, F1-Score e Confusion Matrix.
Infine, combiniamo le previsioni tramite **Majority Vote (Ensemble)**.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix', ax=None):
    cm = confusion_matrix(y_true, y_pred)
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Phishing (-1)', 'Legittimo (1)'],
                yticklabels=['Phishing (-1)', 'Legittimo (1)'])
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predetto')
    ax.set_ylabel('Reale')


models = {
    'Decision Tree':     DecisionTreeClassifier(criterion='entropy', random_state=0),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=0),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
}

predictions = {}
results = []

fig, axes = plt.subplots(1, len(models), figsize=(16, 4))

print('=== VALUTAZIONE DEI MODELLI INDIVIDUALI ===\n')
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_train_scaled, y_train_res)
    y_pred = model.predict(X_test_scaled)
    predictions[name] = y_pred

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec  = recall_score(y_test, y_pred, average='weighted')
    f1   = f1_score(y_test, y_pred, average='weighted')
    results.append({'Modello': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1})

    plot_confusion_matrix(y_test, y_pred, title=f'{name}', ax=ax)
    print(f'--- {name} ---')
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1-Score:  {f1:.4f}\n')

plt.suptitle('Confusion Matrix – Modelli Individuali', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Ensemble – Majority Vote
def majority_vote(predictions_dict):
    preds = np.array(list(predictions_dict.values()))
    majority_votes = []
    for i in range(preds.shape[1]):
        vals, counts = np.unique(preds[:, i], return_counts=True)
        majority_votes.append(vals[np.argmax(counts)])
    return np.array(majority_votes)

y_pred_ensemble = majority_vote(predictions)

acc_ens  = accuracy_score(y_test, y_pred_ensemble)
prec_ens = precision_score(y_test, y_pred_ensemble, average='weighted')
rec_ens  = recall_score(y_test, y_pred_ensemble, average='weighted')
f1_ens   = f1_score(y_test, y_pred_ensemble, average='weighted')
results.append({'Modello': 'Ensemble (Majority Vote)', 'Accuracy': acc_ens, 'Precision': prec_ens, 'Recall': rec_ens, 'F1-Score': f1_ens})

print('--- Ensemble – Majority Vote ---')
print(f'  Accuracy:  {acc_ens:.4f}')
print(f'  Precision: {prec_ens:.4f}')
print(f'  Recall:    {rec_ens:.4f}')
print(f'  F1-Score:  {f1_ens:.4f}')

fig, ax = plt.subplots(figsize=(5, 4))
plot_confusion_matrix(y_test, y_pred_ensemble, title='Ensemble – Majority Vote', ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Tabella riepilogativa dei risultati
results_df = pd.DataFrame(results).set_index('Modello')
print('=== RIEPILOGO METRICHE ===')
print(results_df.to_string(float_format='{:.4f}'.format))

# Barplot comparativo
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
palette = sns.color_palette('muted', len(results_df))

for ax, metric in zip(axes, metrics):
    bars = ax.bar(results_df.index, results_df[metric], color=palette, edgecolor='black', linewidth=0.7)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(0.7, 1.0)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=25)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.002, f'{h:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Confronto Metriche – Tutti i Modelli', fontsize=14)
plt.tight_layout()
plt.show()

## **9. Conclusioni**

### Data Cleaning – Risultati

Il processo di pulizia dei dati ha rappresentato la fase più critica del progetto:

- **Duplicati rimossi**: ~47% delle righe del dataset grezzo erano duplicati, una percentuale altissima che avrebbe distorto gravemente qualsiasi modello. La rimozione ha portato il dataset da 11.055 a 5.849 campioni.
- **Dominio dei valori**: tutte le feature rispettavano il dominio definito nello schema ARFF, con valori in {-1, 0, 1}. Nessuna violazione rilevata.
- **Valori mancanti**: nessun NaN classico presente. I valori `0` nelle feature ternarie sono semanticamente validi.
- **Feature quasi-costanti**: nessuna feature eliminata per varianza insufficiente, il che conferma la qualità informativa del dataset.
- **Correlazioni ridondanti**: nessuna coppia di feature ha superato la soglia di |r| > 0.85.

### Modelli – Risultati

Tutti i modelli raggiungono prestazioni elevate grazie alla buona qualità del dataset pulito e all'applicazione di SMOTE per il bilanciamento:

- Il **Decision Tree** e la **Logistic Regression** mostrano ottime performance bilanciate.
- L'**Ensemble (Majority Vote)** consolida i risultati riducendo la varianza delle predizioni.

### Possibili Miglioramenti

- **Feature Selection automatizzata** (es. Random Forest importance, RFE)
- **Hyperparameter tuning** tramite GridSearchCV o RandomizedSearchCV
- **Modelli più sofisticati**: Random Forest, Gradient Boosting, XGBoost
- **Validazione incrociata** (k-fold) per una stima più robusta delle prestazioni